# Camada Gold — Data Marts e KPIs de Negócio
**Objetivo:** Consolidar os dados Silver em tabelas analíticas prontas para consumo
e responder perguntas-chave via `display()`.

- Projeto 1: Visão Comercial e Volume de Produtos
- Projeto 2: Satisfação de Clientes e Qualidade de Parceiros

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

In [0]:
catalogo = 'olist_catalog'
silver_db_name = 'silver'
gold_db_name = 'gold'

In [0]:
spark.sql(f'USE CATALOG {catalogo}')

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {gold_db_name}')
spark.sql(f'USE SCHEMA {gold_db_name}')

In [0]:
# CARREGAMENTO DAS TABELAS SILVER
 
df_fat_itens    = spark.table(f'{silver_db_name}.fat_itens_pedidos')
df_fat_pedidos  = spark.table(f'{silver_db_name}.fat_pedidos')
df_fat_pag      = spark.table(f'{silver_db_name}.fat_pedido_total')
df_dim_produtos = spark.table(f'{silver_db_name}.dim_produtos')
df_dim_vend     = spark.table(f'{silver_db_name}.dim_vendedores')
df_fat_aval     = spark.table(f'{silver_db_name}.fat_avaliacoes_pedidos')
df_cat_trad     = spark.table(f'{silver_db_name}.dim_categoria_produtos_traducao')
df_cotacao      = spark.table(f'{silver_db_name}.dim_cotacao_dolar')

## 1º Projeto: Visão Comercial e Volume de Produtos

In [0]:
# ENTREGA 1 — gold.fat_vendas_comercial
#
# Granularidade: Ano × Mês × Categoria de Produto
# Métricas: pedidos distintos, itens, receita BRL, receita USD, ticket médio
 
# Base: join itens + pedidos (para datas e status) + produtos (categoria) + cotação
df_base_comercial = (
    df_fat_itens
    .join(df_fat_pedidos.select('id_pedido', 'data_pedido', 'status'), 'id_pedido', 'left')
    .join(df_dim_produtos.select('id_produto', 'categoria_produto'), 'id_produto', 'left')
    # Traduz categoria do produto de PT para EN
    .join(
        df_cat_trad,
        df_dim_produtos.categoria_produto == df_cat_trad.nome_produto_pt,
        'left'
    )
    # Cotação do dólar pelo dia do pedido
    .join(
        df_cotacao,
        F.to_date('data_pedido') == df_cotacao.data_cotacao,
        'left'
    )
    .withColumn('ano_venda', F.year('data_pedido').cast(IntegerType()))
    .withColumn('mes_venda', F.month('data_pedido').cast(IntegerType()))
    # Usa a tradução em inglês quando disponível, senão mantém o original
    .withColumn(
        'categoria_produto',
        F.coalesce(F.col('nome_produto_en'), F.col('categoria_produto'))
    )
)

df_fat_vendas_comercial = (
    df_base_comercial
    .groupBy('ano_venda', 'mes_venda', 'categoria_produto')
    .agg(
        F.countDistinct('id_pedido').alias('total_pedidos'),
        F.count('id_item').alias('qtd_itens_vendidos'),
        F.round(F.sum('preco_BRL'), 2).alias('receita_total_brl'),
        F.round(
            F.sum(F.col('preco_BRL') / F.col('cotacao_compra')), 2
        ).alias('receita_total_usd'),
        F.round(
            F.sum('preco_BRL') / F.countDistinct('id_pedido'), 2
        ).alias('ticket_medio_brl'),
    )
    .orderBy('ano_venda', 'mes_venda', 'categoria_produto')
)

# Modo overwrite conforme especificação
(
    df_fat_vendas_comercial.write
        .format('delta').mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable( f'{gold_db_name}.fat_vendas_comercial')
)
print( f'{gold_db_name}.fat_vendas_comercial: {df_fat_vendas_comercial.count()} linhas.')

In [0]:
# ENTREGA 2 — Rankings Comerciais
#
# Top 5 Produtos MAIS vendidos
# Top 5 Produtos MENOS vendidos
 
# Agrupamento de vendas por produto
df_volume_produto = (
    df_fat_itens
    .join(df_dim_produtos.select('id_produto', 'categoria_produto'), 'id_produto', 'left')
    .groupBy('id_produto', 'categoria_produto')
    .agg(F.count('id_item').alias('quantidade_vendida'))
)

# Top 5 MAIS vendidos
df_top5_mais = (
    df_volume_produto
    .orderBy(F.col('quantidade_vendida').desc())
    .limit(5)
    .select(
        F.col('id_produto').alias('ID Produto'),
        F.col('categoria_produto').alias('Categoria'),
        F.col('quantidade_vendida').alias('Quantidade Vendida')
    )
)

print('Top 5 Produtos MAIS Vendidos:')
display(df_top5_mais)

In [0]:
# Top 5 MENOS vendidos 
df_top5_menos = (
    df_volume_produto
    .orderBy(F.col('quantidade_vendida').asc())
    .limit(5)
    .select(
        F.col('id_produto').alias('ID Produto'),
        F.col('categoria_produto').alias('Categoria'),
        F.col('quantidade_vendida').alias('Quantidade Vendida')
    )
)

print('Top 5 Produtos MENOS Vendidos:')
display(df_top5_menos)

## 2º Projeto: Satisfação de Clientes e Qualidade de Parceiros

In [0]:
# ENTREGA 1 — gold.fat_avaliacoes_clientes
#
# Granularidade: Categoria do Produto × Nome Vendedor × Estado Vendedor
# Métricas: total avaliações, média, positivas (>=4), negativas (<=2), % satisfação
 
# Base: join avaliações + itens (para pegar produto e vendedor) + dimensões
df_base_aval = (
    df_fat_aval
    .join(
        df_fat_itens.select('id_pedido', 'id_produto', 'id_vendedor'),
        'id_pedido', 'left'
    )
    .join(df_dim_produtos.select('id_produto', 'categoria_produto'), 'id_produto', 'left')
    .join(
        df_dim_vend.select('id_vendedor', 'estado'),
        'id_vendedor', 'left'
    )
    .withColumnRenamed('estado', 'estado_vendedor')
)

df_fat_avaliacoes_clientes = (
    df_base_aval
    .groupBy('categoria_produto', 'id_vendedor', 'estado_vendedor')
    .agg(
        F.count('id_avaliacao').alias('total_avaliacoes'),
        F.round(F.avg('nota_avaliacao'), 2).alias('avaliacao_media'),
        F.sum(
            F.when(F.col('nota_avaliacao') >= 4, 1).otherwise(0)
        ).alias('total_avaliacoes_positivas'),
        F.sum(
            F.when(F.col('nota_avaliacao') <= 2, 1).otherwise(0)
        ).alias('total_avaliacoes_negativas'),
    )
    .withColumn(
        # Percentual de satisfação = positivas / total * 100
        'percentual_satisfacao',
        F.round(
            F.col('total_avaliacoes_positivas') / F.col('total_avaliacoes') * 100, 2
        )
    )
)

(
    df_fat_avaliacoes_clientes.write
        .format('delta').mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable( f'{gold_db_name}.fat_avaliacoes_clientes')
)
print( f'{gold_db_name}.fat_avaliacoes_clientes: {df_fat_avaliacoes_clientes.count()} linhas.')

In [0]:
df_fat_avaliacoes_clientes = spark.table(f'{gold_db_name}.fat_avaliacoes_clientes')
display(df_fat_avaliacoes_clientes.limit(10))

In [0]:
# ENTREGA 2 — Rankings de Qualidade (via display)
#
# Critério composto: primeiro nota (desc ou asc), depois volume como desempate (desc)

# Base por produto para rankings
df_aval_produto = (
    df_fat_aval
    .join(df_fat_itens.select('id_pedido', 'id_produto'), 'id_pedido', 'left')
    .join(df_dim_produtos.select('id_produto', 'categoria_produto'), 'id_produto', 'left')
    .groupBy('id_produto', 'categoria_produto')
    .agg(
        F.round(F.avg('nota_avaliacao'), 2).alias('avaliacao_media'),
        F.count('id_avaliacao').alias('total_avaliacoes')
    )
    .filter(F.col('total_avaliacoes') > 5)  # mínimo de avaliações para ranking significativo
)

In [0]:
# 1. Produto MAIS bem avaliado
df_prod_melhor = (
    df_aval_produto
    .orderBy(
        F.col('avaliacao_media').desc(),
        F.col('total_avaliacoes').desc()   # desempate por volume
    )
    .limit(1)
    .select(
        F.col('id_produto').alias('ID Produto'),
        F.col('categoria_produto').alias('Categoria'),
        F.col('avaliacao_media').alias('Avaliação Média'),
        F.col('total_avaliacoes').alias('Total Avaliações')
    )
)
print('Produto MAIS bem avaliado:')
display(df_prod_melhor)

In [0]:
# 2. Produto MENOS bem avaliado
df_prod_pior = (
    df_aval_produto
    .orderBy(
        F.col('avaliacao_media').asc(),
        F.col('total_avaliacoes').desc()   # desempate por volume
    )
    .limit(1)
    .select(
        F.col('id_produto').alias('ID Produto'),
        F.col('categoria_produto').alias('Categoria'),
        F.col('avaliacao_media').alias('Avaliação Média'),
        F.col('total_avaliacoes').alias('Total Avaliações')
    )
)
print('Produto MENOS bem avaliado:')
display(df_prod_pior)

In [0]:
# Base por vendedor para rankings
df_aval_vendedor = (
    df_fat_aval
    .join(df_fat_itens.select('id_pedido', 'id_vendedor'), 'id_pedido', 'left')
    .join(df_dim_vend.select('id_vendedor', 'estado'), 'id_vendedor', 'left')
    .groupBy('id_vendedor', 'estado')
    .agg(
        F.round(F.avg('nota_avaliacao'), 2).alias('avaliacao_media'),
        F.count('id_avaliacao').alias('total_avaliacoes')
    )
    .filter(F.col('total_avaliacoes') > 10)  # critério mínimo para ranking confiável
)

In [0]:
# 3. Vendedor MAIS bem avaliado
df_vend_melhor = (
    df_aval_vendedor
    .orderBy(
        F.col('avaliacao_media').desc(),
        F.col('total_avaliacoes').desc()
    )
    .limit(1)
    .select(
        F.col('id_vendedor').alias('ID Vendedor'),
        F.col('estado').alias('Estado'),
        F.col('avaliacao_media').alias('Avaliação Média'),
        F.col('total_avaliacoes').alias('Total Avaliações')
    )
)
print('Vendedor MAIS bem avaliado:')
display(df_vend_melhor)

In [0]:
# 4. Vendedor MENOS bem avaliado
df_vend_pior = (
    df_aval_vendedor
    .orderBy(
        F.col('avaliacao_media').asc(),
        F.col('total_avaliacoes').desc()
    )
    .limit(1)
    .select(
        F.col('id_vendedor').alias('ID Vendedor'),
        F.col('estado').alias('Estado'),
        F.col('avaliacao_media').alias('Avaliação Média'),
        F.col('total_avaliacoes').alias('Total Avaliações')
    )
)
print('Vendedor MENOS bem avaliado:')
display(df_vend_pior)